# INF0093 - Projeto Prático com Sistemas Multiagentes - 2s 2026
## Prof. Marcelo da Silva Reis
## msreis@unicamp.br

# Aula 2 — Especificação e Baseline
## Estudo de caso: Assistente de Análise de Editais

Nesta primeira versão usamos a solução mais simples: **documento + pergunta → LLM → resposta estruturada**.

Ela servirá como **baseline** para as próximas semanas. Tudo o que for medido aqui será reutilizado
no Entregável 4, quando compararmos as arquiteturas.

## Roteiro

1. Especificação resumida (requisitos, escopo e não-objetivos).
2. Ambiente e registro da execução.
3. Documento de exemplo.
4. Saída estruturada.
5. Baseline instrumentado (latência, tokens, número de chamadas).
6. Conjunto de avaliação **congelado**.
7. Verificação determinística — e por que ela falha se for ingênua.
8. Tabela de resultados e arquivo de referência.
9. Discussão e ponte para o Entregável 2.

## 1. Especificação resumida

**Usuário:** pesquisador preparando uma submissão.

**Entrada:** texto do edital + pergunta.

**Saída:** resposta, evidência e confiança.

**Escopo desta versão:** um edital em português, já em texto, uma pergunta por vez.

**Não-objetivos:** OCR de PDF escaneado, múltiplos editais simultâneos, conversa com histórico,
consulta a fontes externas.

**Premissa:** o documento cabe na janela de contexto do modelo.

| ID | Requisito funcional |
|---|---|
| RF-01 | Identificar prazos. |
| RF-02 | Listar documentos obrigatórios. |
| RF-03 | Identificar critérios de elegibilidade. |
| RF-04 | Responder apenas com base no documento. |
| RF-05 | Indicar explicitamente quando a informação está ausente. |
| RF-06 | Fornecer evidência (trecho literal) que sustente a resposta. |

| ID | Requisito não funcional |
|---|---|
| RNF-01 | Saída estruturada e validada. |
| RNF-02 | Nenhuma afirmação sem evidência presente no documento. |
| RNF-03 | Latência mediana aceitável (aqui, registrada e não imposta). |
| RNF-04 | Custo controlado e mensurado. |
| RNF-05 | Execução reproduzível o suficiente para comparação. |

**Critérios iniciais:** correção, completude, fidelidade da evidência, acerto em informação
ausente, latência, tokens e número de chamadas.

## 2. Ambiente e registro da execução

A célula abaixo funciona tanto no Colab (via `userdata`) quanto localmente (via variável de
ambiente `GROQ_API_KEY`). **Nunca escreva a chave dentro do notebook.**

In [ ]:
%pip install -q -U langchain langchain-groq pydantic pandas

In [ ]:
import os, getpass, datetime, platform

def carregar_chave_groq() -> str:
    """Carrega GROQ_API_KEY do ambiente, do Colab ou do teclado, sem exibi-la."""
    if os.environ.get("GROQ_API_KEY"):
        return "variável de ambiente"
    try:
        from google.colab import userdata          # noqa: F401
        os.environ["GROQ_API_KEY"] = userdata.get("INF0093-2026-2S")
        return "Colab userdata"
    except Exception:
        os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ_API_KEY: ")
        return "entrada manual"

origem = carregar_chave_groq()
assert os.environ.get("GROQ_API_KEY"), "Chave não configurada."
print("Chave carregada via:", origem)

In [ ]:
from langchain_groq import ChatGroq

# Modelos que serão utilizados na correção dos entregáveis.
# Troque a linha ativa para comparar; registre no relatório qual foi usado.
#
# MODEL_NAME = "llama-3.3-70b-versatile"   # Meta
MODEL_NAME = "openai/gpt-oss-20b"          # OpenAI

TEMPERATURE = 0
PROMPT_VERSAO = "v1"

llm = ChatGroq(model=MODEL_NAME, temperature=TEMPERATURE)

# Este registro acompanha os resultados. Sem ele, a comparação do Entregável 4
# fica sem sentido: não saberemos o que mudou entre uma versão e outra.
#
RUN_INFO = {
    "modelo": MODEL_NAME,
    "temperatura": TEMPERATURE,
    "prompt_versao": PROMPT_VERSAO,
    "data": datetime.datetime.now().isoformat(timespec="seconds"),
    "python": platform.python_version(),
}
RUN_INFO

> **Sobre reprodutibilidade (RNF-05):** `temperature=0` reduz a variação, mas **não** garante
> saídas idênticas em serviços de inferência distribuída. Por isso registramos a data e o modelo,
> e por isso vale rodar o conjunto de testes mais de uma vez antes de concluir que uma versão é
> melhor do que outra.

## 3. Documento de exemplo

In [ ]:
call_document = """
CHAMADA PARA PROJETOS DE INOVAÇÃO EM SISTEMAS MULTIAGENTES (AGOSTO DE 2026)

OBJETIVO
Apoiar projetos de inovação tecnológica em sistemas multiagentes, com duração
máxima de 12 meses.

ELEGIBILIDADE
Podem submeter propostas:
- pesquisadores vinculados a universidades brasileiras;
- empresas brasileiras em parceria com uma instituição de pesquisa;
- profissionais com cursos de extensão em sistemas multiagentes.

PRAZO
As propostas devem ser submetidas até 30 de outubro de 2026.

DOCUMENTOS OBRIGATÓRIOS
1. Formulário de submissão;
2. Currículo resumido do coordenador;
3. Plano de trabalho;
4. Orçamento estimado.

RESULTADO
O resultado será divulgado até 15 de dezembro de 2026.
"""

print(f"{len(call_document)} caracteres (~{len(call_document)//4} tokens).")

Repare em duas armadilhas deliberadas:

- **não há** valor de financiamento no documento (testa RF-05);
- a palavra *prazo* é **ambígua**: há um prazo de submissão e uma data de resultado.

## 4. Saída estruturada

A estrutura fixa facilita testes e comparação futura: podemos verificar a resposta, a evidência e
a confiança separadamente.

In [ ]:
from pydantic import BaseModel, Field   # pydantic é uma biblioteca de validação de
                                        # dados, muito utilizada em IA.

from typing import Literal              # Para indicar que uma variável só aceita
                                        # um ou mais valores específicos (literais).

class AnalysisResult(BaseModel):
    answer: str = Field(description="Resposta direta à pergunta.")
    evidence: list[str] = Field(description="Trechos literais do documento que sustentam a resposta.")
    confidence: Literal["high", "medium", "low"]

# include_raw=True devolve também a mensagem original do modelo, de onde extraímos
# a contagem de tokens, e o eventual erro de parsing (em vez de uma exceção).
#
structured_llm = llm.with_structured_output(AnalysisResult, include_raw=True)

print("[done]")

## 5. Baseline instrumentado

O baseline continua sendo **uma única chamada ao modelo**. A única sofisticação é medir o que
custou obtê-la.

In [ ]:
import time

SYSTEM_INSTRUCTION = """
Você é um assistente de análise documental.
Responda exclusivamente com base no documento.
Não invente informações.
Se a informação não estiver presente, a 'answer' deve dizer explicitamente que a informação não
consta no documento, a 'evidence' deve ser uma lista vazia e a 'confidence' deve ser 'low'.
Forneça em 'evidence' apenas trechos copiados literalmente do documento.
Use confiança 'high' para informação explícita, 'medium' para interpretação e 'low' para incerteza.
"""

def analyze_call(question: str, document: str = call_document):
    """Baseline: uma chamada ao LLM. Devolve (resultado, métricas)."""
    prompt = SYSTEM_INSTRUCTION + "\n\nDOCUMENTO:\n" + document + "\n\nPERGUNTA:\n" + question

    inicio = time.perf_counter()
    saida = structured_llm.invoke(prompt)
    latencia = time.perf_counter() - inicio

    uso = getattr(saida["raw"], "usage_metadata", None) or {}
    metricas = {
        "latencia_s": round(latencia, 2),
        "tokens_entrada": uso.get("input_tokens"),
        "tokens_saida": uso.get("output_tokens"),
        "chamadas_llm": 1,
        "erro_parse": str(saida["parsing_error"]) if saida["parsing_error"] else None,
    }
    return saida["parsed"], metricas

print("[done]")

## 6. Primeira execução

In [ ]:
resultado, metricas = analyze_call("Qual é o prazo para submissão?")
print(resultado.model_dump())
print(metricas)

In [ ]:
# Caso de interpretação: a resposta exige ler a lista de elegibilidade com atenção.
#
resultado, metricas = analyze_call("Profissionais em aprendizado de máquina clássico são elegíveis?")
print("RESPOSTA :", resultado.answer)
print("EVIDÊNCIA:", resultado.evidence)
print("CONFIANÇA:", resultado.confidence)
print("MÉTRICAS :", metricas)

## 7. Conjunto de avaliação congelado

Este é o ativo mais valioso do projeto. Ele é escrito **antes** de ajustarmos o prompt e é
**congelado**: os mesmos casos serão executados nos Entregáveis 2, 3 e 4.

Cobrimos quatro tipos de caso: normal, lista (completude), interpretação e informação ausente.
Um quinto caso é ambíguo de propósito e fica marcado para avaliação manual.

> Com apenas 4 casos automáticos, um acerto vale 25 pontos percentuais. Para o projeto de vocês,
> 10 a 20 casos dão muito mais poder de comparação.

In [ ]:
test_cases = [
    {"id": "T01", "tipo": "normal", "verificacao": "auto",
     "pergunta": "Qual é o prazo para submissão?",
     "esperado": ["30 de outubro de 2026"], "cobertura_minima": 1.0},

    {"id": "T02", "tipo": "lista", "verificacao": "auto",
     "pergunta": "Quais documentos são obrigatórios?",
     "esperado": ["formulário", "currículo", "plano de trabalho", "orçamento"],
     "cobertura_minima": 1.0},

    {"id": "T03", "tipo": "interpretação", "verificacao": "auto",
     "pergunta": "Quem pode participar?",
     "esperado": ["universidades brasileiras", "empresas brasileiras"],
     "cobertura_minima": 0.5},

    {"id": "T04", "tipo": "informação ausente", "verificacao": "auto",
     "pergunta": "Qual é o valor máximo de financiamento?",
     "esperado": None},

    {"id": "T05", "tipo": "ambíguo", "verificacao": "manual",
     "pergunta": "Qual é o prazo?",
     "esperado": None,
     "nota": "Esperado: pedir esclarecimento ou distinguir submissão (30/10) de resultado (15/12)."},
]

print(len(test_cases), "casos;",
      sum(c["verificacao"] == "auto" for c in test_cases), "automáticos.")

## 8. Verificação determinística

Quatro verificações, cada uma ligada a um requisito:

| Verificação | Requisito | O que mede |
|---|---|---|
| `cobertura_esperada` | RF-01/02/03 | correção e completude da resposta |
| `admite_ausencia` | RF-05 | o sistema sabe dizer "não sei" |
| `evidencia_fiel` | RF-06 / RNF-02 | a evidência citada existe mesmo no documento |
| `erro_parse` | RNF-01 | a saída respeitou o esquema |

A normalização (minúsculas, sem acentos) evita reprovar uma resposta correta por causa de um
acento. Continua sendo frágil a paráfrases — voltamos a isso na seção 9.

In [ ]:
import re, unicodedata
from typing import Optional

def normalizar(texto: str) -> str:
    texto = unicodedata.normalize("NFKD", texto.lower())
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", texto).strip()

MARCADORES_AUSENCIA = [
    "nao esta", "nao consta", "nao foi encontrad", "nao encontrei", "nao informad",
    "nao especificad", "nao ha informacao", "nao menciona", "nao e mencionad",
    "ausente no documento", "nao aparece", "nao define", "nao indica",
]

def admite_ausencia(resultado: AnalysisResult) -> bool:
    """RF-05: não basta escrever 'não'; a saída inteira precisa ser coerente com a abstenção."""
    texto = normalizar(resultado.answer)
    declarou = any(m in texto for m in MARCADORES_AUSENCIA)
    return declarou and len(resultado.evidence) == 0 and resultado.confidence == "low"

def cobertura_esperada(resultado: AnalysisResult, esperado: list[str]) -> float:
    """Fração dos itens de referência presentes na resposta (correção + completude)."""
    texto = normalizar(resultado.answer)
    achados = [k for k in esperado if normalizar(k) in texto]
    return len(achados) / len(esperado)

def evidencia_fiel(resultado: AnalysisResult, documento: str) -> Optional[float]:
    """RNF-02: fração das evidências citadas que realmente ocorrem no documento."""
    if not resultado.evidence:
        return None
    doc = normalizar(documento)
    validas = [e for e in resultado.evidence if normalizar(e) in doc]
    return len(validas) / len(resultado.evidence)

def avaliar(caso: dict, resultado: AnalysisResult) -> dict:
    if caso["verificacao"] == "manual":
        return {"aprovado": None, "cobertura": None}
    if caso["esperado"] is None:
        return {"aprovado": admite_ausencia(resultado), "cobertura": None}
    cobertura = cobertura_esperada(resultado, caso["esperado"])
    return {"aprovado": cobertura >= caso.get("cobertura_minima", 1.0),
            "cobertura": round(cobertura, 2)}

print("[done]")

## 9. Por que a verificação ingênua falha

Uma tentação comum é testar informação ausente procurando a palavra `"não"` na resposta.
A célula abaixo mostra por que isso não funciona — e **não precisa do LLM** para mostrar.

In [ ]:
# Resposta fabricada à mão: nega uma coisa e alucina outra.
#
resposta_ruim = AnalysisResult(
    answer="O documento não define limite de páginas, mas o valor máximo de financiamento é de R$ 500.000.",
    evidence=["O valor máximo de financiamento é de R$ 500.000."],
    confidence="high",
)

verificacao_ingenua = "nao" in normalizar(resposta_ruim.answer)

print("Verificação ingênua ('não' na resposta):", verificacao_ingenua)   # True  -> falso positivo
print("admite_ausencia(...)                   :", admite_ausencia(resposta_ruim))
print("evidencia_fiel(...)                    :", evidencia_fiel(resposta_ruim, call_document))

A verificação ingênua **aprova** uma resposta que inventa um valor. As outras duas reprovam.

Lição para o Entregável 1: um critério de sucesso mal escrito produz números bonitos e conclusões
erradas. Quando a verificação determinística não der conta (respostas abertas, redação livre),
use rubrica humana ou LLM como juiz — e diga qual foi usada.

## 10. Execução do conjunto

In [ ]:
registros = []

for caso in test_cases:
    resultado, metricas = analyze_call(caso["pergunta"])
    nota = avaliar(caso, resultado)

    registros.append({
        "id": caso["id"],
        "tipo": caso["tipo"],
        "pergunta": caso["pergunta"],
        "resposta": resultado.answer,
        "confianca": resultado.confidence,
        "n_evidencias": len(resultado.evidence),
        "evidencia_fiel": evidencia_fiel(resultado, call_document),
        **nota,
        **metricas,
    })

    print("=" * 78)
    print(f'[{caso["id"]}] ({caso["tipo"]}) {caso["pergunta"]}')
    print("RESPOSTA :", resultado.answer)
    print("EVIDÊNCIA:", resultado.evidence)
    print("CONFIANÇA:", resultado.confidence)
    print("APROVADO :", nota["aprovado"], "| latência:", metricas["latencia_s"], "s")

In [ ]:
import pandas as pd

df = pd.DataFrame(registros)
df[["id", "tipo", "aprovado", "cobertura", "evidencia_fiel",
    "confianca", "latencia_s", "tokens_entrada", "tokens_saida"]]

In [ ]:
# Preços por milhão de tokens. CONFIRA os valores vigentes em https://groq.com/pricing
# antes de reportar custo no entregável.
#
PRECO_USD_POR_MILHAO = {"entrada": 0.10, "saida": 0.50}

autos = df[df["aprovado"].notna()]
tokens_in = df["tokens_entrada"].fillna(0).sum()
tokens_out = df["tokens_saida"].fillna(0).sum()

custo = (tokens_in * PRECO_USD_POR_MILHAO["entrada"]
         + tokens_out * PRECO_USD_POR_MILHAO["saida"]) / 1e6

RESUMO = {
    "casos_automaticos": int(len(autos)),
    "taxa_aprovacao": round(float(autos["aprovado"].astype(bool).mean()), 2) if len(autos) else None,
    "latencia_mediana_s": round(float(df["latencia_s"].median()), 2),
    "chamadas_llm": int(df["chamadas_llm"].sum()),
    "tokens_entrada": int(tokens_in),
    "tokens_saida": int(tokens_out),
    "custo_estimado_usd": round(float(custo), 6),
}
RESUMO

In [ ]:
import json

# Este arquivo é a referência do baseline. Guarde-o junto com o notebook:
# nos próximos entregáveis, a nova arquitetura será comparada com estes números.
#
referencia = {"run": RUN_INFO, "resumo": RESUMO, "registros": registros}

with open("baseline_v1_resultados.json", "w", encoding="utf-8") as f:
    json.dump(referencia, f, ensure_ascii=False, indent=2, default=str)

print("Salvo em baseline_v1_resultados.json")

## 11. Discussão

1. O que acontece com um edital de 100 páginas?
2. Como lidar com anexos e múltiplos documentos?
3. Como consultar informações externas (por exemplo, um prazo prorrogado no site da agência)?
4. Como manter uma conversa longa sobre o mesmo edital?
5. Como decompor perguntas compostas ("quem pode participar e até quando?")?
6. Quando especialistas separados seriam úteis?
7. O ganho de qualidade compensaria custo e latência?

### Ponte para as próximas semanas

- tarefa composta → **workflow/ReAct**;
- conversa → **memória**;
- fontes externas → **ferramentas/MCP**;
- especialização → **MAS/skills**;
- decomposição dinâmica → **planejamento**;
- falhas → **tratamento de erros**;
- comparação com esta versão → **avaliação arquitetural**.

## 12. Exercício para os grupos

Refaça este notebook para o problema escolhido pelo grupo:

1. especifique o problema, o escopo e os não-objetivos;
2. escreva requisitos **verificáveis** (RF e RNF);
3. escreva o conjunto de avaliação **antes** de ajustar o prompt (mínimo 3 casos; recomendado 10);
4. implemente o baseline mais simples possível e classifique-o (completo / parcial / proxy);
5. instrumente a execução (latência, tokens, chamadas) e salve o arquivo de referência;
6. registre limitações e responda:

> **Como poderemos demonstrar, ao final do curso, que a arquitetura final é melhor do que este baseline?**